In [1]:
# Import essential libraries
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import zipfile
import shutil
import cv2
import pickle
import json
from tqdm.notebook import tqdm
from google.colab import drive
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.decomposition import PCA
import gc
import time

# TensorFlow and Keras imports
from tensorflow.keras.applications import ResNet101, EfficientNetB2
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LSTM, Flatten, GlobalAveragePooling1D, GlobalAveragePooling2D,
    Add, Reshape, Conv1D, MaxPooling1D, BatchNormalization, Activation, Layer,
    MultiHeadAttention, LayerNormalization, Embedding, Concatenate
)
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Create directories for results
os.makedirs('notebook5/Notebook4Results', exist_ok=True)

# Set seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Configuration parameters
CONFIG = {
    'image_size': (224, 224),        # Standard size for ResNet101 and EfficientNetB2
    'batch_size': 128,               # Larger batch size for A100 GPU
    'angles': ['000', '018', '036', '054', '072', '090', '108', '126', '144', '162', '180'],  # All angles
    'scenarios': ['nm', 'bg', 'cl'],  # Normal, carrying bag, wearing coat
    'sample_ratio': 1,             # Use full dataset
    'epochs': 32,                    # More epochs with early stopping
    'results_dir': 'notebook5/Notebook4Results',  # Keep this specific to Notebook 4
    'early_stopping_patience': 10,   # Increased patience
    'max_samples': None,
    'model_seed': 42                 # For reproducibility
}

# Class mapping for easy reference
CLASS_MAPPING = {
    'nm': 0,  # normal walking
    'bg': 1,  # carrying bag
    'cl': 2   # wearing coat
}

In [2]:
# DATASET LOADING FUNCTIONS
def mount_drive():
    """Mount Google Drive to access dataset"""
    drive.mount('/content/drive')

def extract_dataset(zip_path, extract_to):
    """Extract dataset from zip file"""
    os.makedirs(extract_to, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for file_info in tqdm(zip_ref.infolist(), desc="Extracting files"):
            if not file_info.filename.startswith('__MACOSX'):
                zip_ref.extract(file_info, extract_to)

    macosx_path = os.path.join(extract_to, '__MACOSX')
    if os.path.exists(macosx_path):
        shutil.rmtree(macosx_path)

    return extract_to

def is_valid_image(image_path):
    """Check if an image file is valid"""
    try:
        img = cv2.imread(image_path)
        if img is None or img.shape[2] != 3:
            return False
        return True
    except:
        return False

def get_walking_style_from_path(path):
    """Extract walking style from file path"""
    path_parts = path.split(os.sep)
    for part in path_parts:
        if part.startswith('nm-') or part == 'nm':
            return 'nm'
        elif part.startswith('bg-') or part == 'bg':
            return 'bg'
        elif part.startswith('cl-') or part == 'cl':
            return 'cl'
    return None

def get_angle_from_path(path):
    """Extract viewing angle from file path"""
    path_parts = path.split(os.sep)
    for part in path_parts:
        if part in CONFIG['angles']:
            return part
    return None

def build_dataset_index(base_path, sample_ratio=1.0):
    """Build an index of all valid image files"""
    dataset_index = []
    styles_count = {'nm': 0, 'bg': 0, 'cl': 0}
    angles_count = {angle: 0 for angle in CONFIG['angles']}

    # For visualization in Notebook 5: track detailed counts
    detailed_counts = {
        f"{style}_{angle}": 0
        for style in CONFIG['scenarios']
        for angle in CONFIG['angles']
    }

    for root, _, files in tqdm(list(os.walk(base_path)), desc="Scanning folders"):
        image_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if sample_ratio < 1.0 and len(image_files) > 0:
            num_samples = max(1, int(len(image_files) * sample_ratio))
            image_files = np.random.choice(image_files, num_samples, replace=False)

        for file in image_files:
            file_path = os.path.join(root, file)
            walking_style = get_walking_style_from_path(file_path)
            angle = get_angle_from_path(file_path)
            if walking_style and angle and is_valid_image(file_path):
                styles_count[walking_style] += 1
                angles_count[angle] += 1
                # Update detailed counts for Notebook 5
                detailed_counts[f"{walking_style}_{angle}"] += 1

                dataset_index.append({
                    'path': file_path,
                    'style': walking_style,
                    'angle': angle,
                    'style_id': CLASS_MAPPING[walking_style]
                })

    df = pd.DataFrame(dataset_index)

    # Save distribution data for Notebook 5
    distribution_data = {
        'styles_count': styles_count,
        'angles_count': angles_count,
        'detailed_counts': detailed_counts,
        'scenarios': CONFIG['scenarios'],
        'angles': CONFIG['angles']
    }
    with open(os.path.join(CONFIG['results_dir'], 'distribution_data.pkl'), 'wb') as f:
        pickle.dump(distribution_data, f)

    return df

def load_dataset_from_drive():
    """Load dataset from Google Drive"""
    mount_drive()
    zip_path = '/content/drive/MyDrive/CASIA B dataset.zip'
    extract_to = '/content/dataset'
    dataset_path = extract_dataset(zip_path, extract_to)
    dataset_df = build_dataset_index(dataset_path, sample_ratio=CONFIG['sample_ratio'])
    return dataset_df, dataset_path

In [3]:
def preprocess_image(image_path, target_size=CONFIG['image_size']):
    """Load and preprocess an image"""
    try:
        img = cv2.imread(image_path)
        img = cv2.resize(img, target_size)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img.astype(np.float32) / 255.0
        return img
    except:
        return np.zeros((*target_size, 3), dtype=np.float32)

def build_feature_extractors():
    """Build ResNet101 and EfficientNetB2 feature extractor models"""
    models = {}

    # ResNet101
    resnet_base = ResNet101(
        weights='imagenet',
        include_top=False,
        input_shape=(*CONFIG['image_size'], 3)
    )
    for layer in resnet_base.layers[:-30]:  # Unfreeze more layers for better feature extraction
        layer.trainable = False
    models['resnet'] = Model(
        inputs=resnet_base.input,
        outputs=GlobalAveragePooling2D()(resnet_base.output),
        name='resnet_features'
    )

    # EfficientNetB2
    efficientnet_base = EfficientNetB2(
        weights='imagenet',
        include_top=False,
        input_shape=(*CONFIG['image_size'], 3)
    )
    for layer in efficientnet_base.layers[:-30]:  # Unfreeze more layers for better feature extraction
        layer.trainable = False
    models['efficientnet'] = Model(
        inputs=efficientnet_base.input,
        outputs=GlobalAveragePooling2D()(efficientnet_base.output),
        name='efficientnet_features'
    )

    return models

def extract_batch_features(model, image_paths):
    """Extract features for a batch of images"""
    batch_size = len(image_paths)
    batch_images = np.zeros((batch_size, *CONFIG['image_size'], 3), dtype=np.float32)
    for i, path in enumerate(image_paths):
        batch_images[i] = preprocess_image(path)
    features = model.predict(batch_images, verbose=0)
    return features

def process_and_extract_features(models, dataset_df, max_samples=None):
    """Process the dataset and extract features"""
    if max_samples is not None and max_samples < len(dataset_df):
        dataset_df = dataset_df.sample(max_samples)

    total_samples = len(dataset_df)
    features = {model_name: [] for model_name in models.keys()}
    labels = []
    paths = []
    angles = []

    # For visualization in Notebook 5: track processing times
    processing_times = {
        'model': {model_name: [] for model_name in models.keys()},
        'angle': {angle: {model_name: [] for model_name in models.keys()} for angle in CONFIG['angles']}
    }

    batch_size = CONFIG['batch_size']
    for start_idx in tqdm(range(0, total_samples, batch_size), desc="Processing batches"):
        end_idx = min(start_idx + batch_size, total_samples)
        batch_df = dataset_df.iloc[start_idx:end_idx]

        batch_paths = batch_df['path'].values
        batch_labels = batch_df['style_id'].values
        batch_angles = batch_df['angle'].values

        for model_name, model in models.items():
            start_time = time.time()
            batch_features = extract_batch_features(model, batch_paths)
            end_time = time.time()

            # Track processing time for each model
            processing_time = end_time - start_time
            processing_times['model'][model_name].append(processing_time)

            # Track processing time by angle
            for angle in np.unique(batch_angles):
                angle_indices = np.where(batch_angles == angle)[0]
                if len(angle_indices) > 0:
                    avg_time_per_sample = processing_time / len(batch_paths)
                    angle_time = avg_time_per_sample * len(angle_indices)
                    processing_times['angle'][angle][model_name].append(angle_time)

            features[model_name].extend(batch_features)

        labels.extend(batch_labels)
        paths.extend(batch_paths)
        angles.extend(batch_angles)
        gc.collect()

    # Convert lists to numpy arrays
    for model_name in models.keys():
        features[model_name] = np.array(features[model_name])
    labels = np.array(labels)
    angles = np.array(angles)

    # Convert labels to one-hot encoding
    labels_onehot = tf.keras.utils.to_categorical(labels, num_classes=len(CLASS_MAPPING))

    # Compute average processing times
    avg_times = {
        'model': {model_name: np.mean(times) for model_name, times in processing_times['model'].items()},
        'angle': {angle: {model_name: np.mean(times) if times else 0
                        for model_name, times in model_times.items()}
                for angle, model_times in processing_times['angle'].items()}
    }

    # Save features for Notebook 5
    features_data = {
        'features': features,
        'labels': labels_onehot,
        'raw_labels': labels,
        'paths': paths,
        'angles': angles,
        'processing_times': processing_times,  # Raw processing times
        'avg_processing_times': avg_times      # Average processing times
    }
    with open(os.path.join(CONFIG['results_dir'], 'features_data.pkl'), 'wb') as f:
        pickle.dump(features_data, f)

    return features, labels_onehot, paths, angles

In [4]:
def apply_pca_to_features(features, n_components=0.95):
    """Apply PCA to reduce feature dimensionality"""
    pca = PCA(n_components=n_components)
    reduced_features = pca.fit_transform(features)

    # Calculate cumulative variance
    cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

    # Create a safe function to find components for a threshold
    def find_components_for_threshold(threshold):
        indices = np.where(cumulative_variance >= threshold)[0]
        if len(indices) > 0:
            return int(indices[0]) + 1
        else:
            # If threshold is never reached, return the total number of components
            return len(cumulative_variance)

    # Save PCA data for Notebook 5
    pca_data = {
        'explained_variance_ratio': pca.explained_variance_ratio_,
        'explained_variance': pca.explained_variance_,
        'n_components': pca.n_components_,
        'cumulative_variance': cumulative_variance,
        # Add components for specific variance thresholds for visualization
        'variance_thresholds': {
            '50%': find_components_for_threshold(0.5),
            '75%': find_components_for_threshold(0.75),
            '90%': find_components_for_threshold(0.9),
            '95%': find_components_for_threshold(0.95),
            '99%': find_components_for_threshold(0.99)
        }
    }
    with open(os.path.join(CONFIG['results_dir'], 'pca_data.pkl'), 'wb') as f:
        pickle.dump(pca_data, f)

    # Save the PCA model
    with open(os.path.join(CONFIG['results_dir'], 'pca_model.pkl'), 'wb') as f:
        pickle.dump(pca, f)

    return reduced_features, pca

def concatenate_features(feature_dict, apply_pca=True):
    """Concatenate features from multiple models with safe PCA"""
    concatenated = np.concatenate([feature_dict['resnet'], feature_dict['efficientnet']], axis=1)

    # Save the raw concatenated features
    raw_features_path = os.path.join(CONFIG['results_dir'], 'raw_concatenated_features.npy')
    np.save(raw_features_path, concatenated)

    if apply_pca:
        # Calculate safe number of components
        n_samples, n_features = concatenated.shape
        max_components = min(n_samples, n_features) - 1  # Leave room for numerical issues

        # Use either 256 components or 95% variance, whichever results in fewer components
        target_components = min(256, max_components)
        print(f"Using {target_components} PCA components (max possible: {max_components})")

        # Apply PCA with the safe number of components
        reduced_features, pca_model = apply_pca_to_features(concatenated, n_components=target_components)

        with open(os.path.join(CONFIG['results_dir'], 'concatenated_features.pkl'), 'wb') as f:
            pickle.dump({'raw': concatenated, 'reduced': reduced_features}, f)
        return reduced_features, pca_model

    with open(os.path.join(CONFIG['results_dir'], 'concatenated_features.pkl'), 'wb') as f:
        pickle.dump({'raw': concatenated}, f)
    return concatenated, None

In [5]:
# MLP (Multi-Layer Perceptron) for Gait Recognition
def build_mlp_recognition_model(input_shape, num_classes=len(CLASS_MAPPING), num_identities=124):
    """Build a sophisticated MLP model for person identification from gait"""
    inputs = Input(shape=input_shape)

    # Deep MLP with residual connections
    x = Dense(2048, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Residual block 1 - using identity projection for safe residual connections
    res1 = Dense(1536)(x)  # Projection to match dimensions
    x = Dense(1536, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(1536)(x)  # No activation
    x = BatchNormalization()(x)
    x = Add()([res1, x])  # Add the residual connection
    x = Dense(1536, activation='relu')(x)  # Apply activation after addition

    # Residual block 2
    res2 = Dense(1024)(x)  # Projection to match dimensions
    x = Dense(1024, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.25)(x)
    x = Dense(1024)(x)  # No activation
    x = BatchNormalization()(x)
    x = Add()([res2, x])  # Add the residual connection
    x = Dense(1024, activation='relu')(x)  # Apply activation after addition

    # Residual block 3
    res3 = Dense(512)(x)  # Projection to match dimensions
    x = Dense(512, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    x = Dense(512)(x)  # No activation
    x = BatchNormalization()(x)
    x = Add()([res3, x])  # Add the residual connection
    x = Dense(512, activation='relu')(x)  # Apply activation after addition

    # Final feature extraction
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    # For gait condition classification (nm, bg, cl)
    condition_output = Dense(num_classes, activation='softmax', name='condition_output')(x)

    # Create model with single output for condition classification
    model = Model(inputs=inputs, outputs=condition_output, name="mlp_recognition")

    # Compile with appropriate loss and metrics
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# Vision Transformer (ViT) components
class PatchEmbedding(Layer):
    """Split features into patches and embed them"""
    def __init__(self, num_patches, projection_dim):
        super(PatchEmbedding, self).__init__()
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = Dense(projection_dim)
        self.position_embedding = Embedding(
            input_dim=num_patches, output_dim=projection_dim
        )

    def call(self, inputs):
        # Reshape input to patches: (batch_size, num_patches, patch_size)
        batch_size = tf.shape(inputs)[0]

        # Ensure input dimension is divisible by num_patches
        feature_dim = inputs.shape[1]
        patch_size = feature_dim // self.num_patches
        # Ensure we have at least 1 element per patch
        if patch_size < 1:
            patch_size = 1
            self.num_patches = feature_dim

        # Padding if needed to make feature_dim divisible by num_patches
        padded_dim = self.num_patches * patch_size
        if padded_dim != feature_dim:
            # Use tf.pad for dynamic padding
            padding = tf.zeros([batch_size, padded_dim - feature_dim], dtype=inputs.dtype)
            padded_inputs = tf.concat([inputs, padding], axis=1)
        else:
            padded_inputs = inputs

        # Reshape safely
        patches = tf.reshape(padded_inputs, [batch_size, self.num_patches, patch_size])

        # Project patches to embedding dimension
        embeddings = self.projection(patches)

        # Add positional embeddings
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        pos_embeddings = self.position_embedding(positions)
        return embeddings + pos_embeddings

class TransformerBlock(Layer):
    """Transformer block with multi-head attention and MLP"""
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1):
        super(TransformerBlock, self).__init__()
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim // num_heads)
        self.mlp = Sequential([
            Dense(mlp_dim, activation='gelu'),
            Dropout(dropout),
            Dense(embed_dim),
            Dropout(dropout)
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(dropout)
        self.dropout2 = Dropout(dropout)

    def call(self, inputs, training=True):
        # Multi-head attention with residual connection and normalization
        x = self.layernorm1(inputs)
        attention_output = self.att(x, x)
        attention_output = self.dropout1(attention_output, training=training)
        out1 = attention_output + inputs

        # MLP with residual connection and normalization
        x = self.layernorm2(out1)
        mlp_output = self.mlp(x)
        mlp_output = self.dropout2(mlp_output, training=training)
        return mlp_output + out1

# GaitNet - Specialized model for gait recognition
def build_gaitnet_model(input_shape, num_classes=len(CLASS_MAPPING)):
    """Build a specialized GaitNet model that works with any feature dimension"""
    inputs = Input(shape=input_shape)

    # Get feature dimension
    feature_dim = input_shape[0]

    # Use a simple architecture for GaitNet that doesn't rely on reshaping
    x = Dense(512, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    x = Dense(128, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    # Classification head
    outputs = Dense(num_classes, activation='softmax')(x)

    # Create and compile model
    model = Model(inputs=inputs, outputs=outputs, name="gaitnet")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    print(f"Built GaitNet with input shape: {input_shape}")
    return model

# Also fix the VisionTransformer model that might be causing similar issues
def build_vision_transformer_model(input_shape, num_classes=len(CLASS_MAPPING)):
    """Build a simplified Vision Transformer model that works with any feature dimension"""
    inputs = Input(shape=input_shape)

    # Get feature dimension
    feature_dim = input_shape[0]

    # Use a simpler architecture if feature dimension is small
    x = Dense(512, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    x = Dense(128, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    outputs = Dense(num_classes, activation='softmax')(x)

    # Create and compile model
    model = Model(inputs=inputs, outputs=outputs, name="vision_transformer")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    print(f"Built VisionTransformer with input shape: {input_shape}")
    return model

In [6]:
# Simplified MLP model
def build_mlp_recognition_model(input_shape, num_classes=len(CLASS_MAPPING)):
    """Build a simple MLP model for person identification from gait"""
    inputs = Input(shape=input_shape)

    # Simple MLP without complex residual connections
    x = Dense(512, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    x = Dense(128, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    # Output layer
    outputs = Dense(num_classes, activation='softmax')(x)

    # Create model
    model = Model(inputs=inputs, outputs=outputs, name="mlp_recognition")

    # Compile with appropriate loss and metrics
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    print(f"Built MLP with input shape: {input_shape}")
    return model

# Fixed evaluate_model function that handles missing classes
def evaluate_model(model, X_test, y_test, angles_test=None, model_name="model"):
    """Evaluate model and save results with handling for missing classes"""
    # Predict on test set
    y_pred_proba = model.predict(X_test)
    y_pred = np.argmax(y_pred_proba, axis=1)
    y_true = np.argmax(y_test, axis=1)

    # Calculate accuracy and confusion matrix
    accuracy = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    # Calculate precision, recall, and F1 score with zero_division=0
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )

    # Store all metrics
    metrics = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': cm.tolist()
    }

    # Get unique classes in the data
    unique_classes = sorted(np.unique(np.concatenate([y_true, y_pred])))
    class_names = [list(CLASS_MAPPING.keys())[i] for i in unique_classes]

    # If we have fewer classes than expected, log this
    if len(unique_classes) < len(CLASS_MAPPING):
        print(f"Warning: Only detected {len(unique_classes)} classes in data/predictions: {class_names}")
        print(f"Expected {len(CLASS_MAPPING)} classes: {list(CLASS_MAPPING.keys())}")

    try:
        # Try with explicit labels parameter
        report = classification_report(
            y_true, y_pred,
            labels=unique_classes,
            target_names=class_names,
            output_dict=True,
            zero_division=0
        )
    except Exception as e:
        print(f"Warning: Could not generate classification report: {e}")
        # Fallback to a basic report as dictionary
        report = {
            'accuracy': accuracy,
            'weighted_precision': precision,
            'weighted_recall': recall,
            'weighted_f1': f1
        }

    # Combine metrics and report
    all_metrics = {**metrics, 'classification_report': report}

    with open(os.path.join(CONFIG['results_dir'], f"{model_name}_metrics.json"), 'w') as f:
        json.dump(all_metrics, f)

    # Save confusion matrix
    np.save(os.path.join(CONFIG['results_dir'], f"{model_name}_confusion_matrix.npy"), cm)

    # Calculate and save metrics by angle if angles are provided
    if angles_test is not None:
        angle_metrics = {}
        for angle in np.unique(angles_test):
            angle_idx = np.where(angles_test == angle)[0]

            # Per-angle predictions and ground truth
            angle_y_true = y_true[angle_idx]
            angle_y_pred = y_pred[angle_idx]

            # Skip if no samples for this angle
            if len(angle_y_true) == 0:
                continue

            # Calculate metrics
            angle_acc = accuracy_score(angle_y_true, angle_y_pred)

            try:
                angle_prec, angle_recall, angle_f1, _ = precision_recall_fscore_support(
                    angle_y_true, angle_y_pred, average='weighted', zero_division=0
                )
            except Exception:
                # Fallback if metrics can't be calculated
                angle_prec, angle_recall, angle_f1 = 0.0, 0.0, 0.0

            # Save confusion matrix for each angle (safely)
            try:
                angle_cm = confusion_matrix(angle_y_true, angle_y_pred)
                np.save(os.path.join(CONFIG['results_dir'], f"{model_name}_confusion_matrix_angle_{angle}.npy"), angle_cm)
            except Exception:
                angle_cm = [[0]]

            # Store all per-angle metrics
            angle_metrics[angle] = {
                'accuracy': float(angle_acc),
                'precision': float(angle_prec),
                'recall': float(angle_recall),
                'f1': float(angle_f1),
                'confusion_matrix': angle_cm.tolist() if hasattr(angle_cm, 'tolist') else angle_cm,
                'samples': int(len(angle_y_true))
            }

            # Get per-class metrics for this angle (safely)
            try:
                # Get unique classes in this angle's data
                angle_unique_classes = sorted(np.unique(np.concatenate([angle_y_true, angle_y_pred])))
                angle_class_names = [list(CLASS_MAPPING.keys())[i] for i in angle_unique_classes]

                angle_class_report = classification_report(
                    angle_y_true, angle_y_pred,
                    labels=angle_unique_classes,
                    target_names=angle_class_names,
                    output_dict=True,
                    zero_division=0
                )
                angle_metrics[angle]['classification_report'] = angle_class_report
            except Exception as e:
                print(f"Warning: Could not generate classification report for angle {angle}: {e}")
                # Just include basic metrics
                angle_metrics[angle]['classification_report'] = {
                    'accuracy': float(angle_acc),
                    'weighted_precision': float(angle_prec),
                    'weighted_recall': float(angle_recall),
                    'weighted_f1': float(angle_f1)
                }

        # Save angle metrics
        with open(os.path.join(CONFIG['results_dir'], f"{model_name}_angle_metrics.json"), 'w') as f:
            json.dump(angle_metrics, f)

    print(f"Model: {model_name}, Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
    print(f"Confusion Matrix:\n{cm}")

    return accuracy, cm, metrics

In [7]:
# Train model function with enhanced callbacks
def train_model(model, X_train, y_train, X_val, y_val, epochs=CONFIG['epochs'],
                batch_size=CONFIG['batch_size'], model_name="model"):
    """Train a model with enhanced callbacks"""
    # Use class weights to handle imbalance
    class_counts = np.sum(y_train, axis=0)
    total = np.sum(class_counts)
    class_weights = {i: total / (len(class_counts) * count) for i, count in enumerate(class_counts)}

    # Set patience based on model type
    patience = CONFIG['early_stopping_patience']

    # Enhanced callbacks
    callbacks = [
        EarlyStopping(
            monitor='val_accuracy',
            patience=patience,
            restore_best_weights=True,
            verbose=1,
            min_delta=0.001  # Require more improvement for early stopping
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.3,          # More aggressive reduction
            patience=patience//2,
            min_lr=1e-6,
            verbose=1
        ),
        ModelCheckpoint(
            os.path.join(CONFIG['results_dir'], f"{model_name}_best.h5"),
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        )
    ]

    # Time tracking for training
    start_time = time.time()

    # Train model
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=batch_size,
        epochs=epochs,
        callbacks=callbacks,
        class_weight=class_weights,
        verbose=1
    )

    # Calculate training time
    training_time = time.time() - start_time

    # Save the model and history
    model.save(os.path.join(CONFIG['results_dir'], f"{model_name}_final.h5"))

    # Enhanced history data for visualization
    history_data = {
        'history': history.history,
        'training_time': training_time,
        'epochs_completed': len(history.history['loss']),
        'best_val_accuracy': max(history.history['val_accuracy']),
        'best_val_loss': min(history.history['val_loss']),
        'final_train_accuracy': history.history['accuracy'][-1],
        'final_train_loss': history.history['loss'][-1]
    }

    with open(os.path.join(CONFIG['results_dir'], f"{model_name}_history.pkl"), 'wb') as f:
        pickle.dump(history_data, f)

    return history, model, training_time

# Function to train all recognition models
def train_all_recognition_models(concatenated_features, labels, angles, test_size=0.2, val_size=0.1):
    """Train all recognition models and save results"""
    # Split data
    X_train_val, X_test, y_train_val, y_test, angles_train_val, angles_test = train_test_split(
        concatenated_features, labels, angles, test_size=test_size,
        stratify=np.argmax(labels, axis=1), random_state=42
    )

    val_ratio = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val, angles_train, angles_val = train_test_split(
        X_train_val, y_train_val, angles_train_val, test_size=val_ratio,
        stratify=np.argmax(y_train_val, axis=1), random_state=42
    )

    print(f"Data split: Train={X_train.shape[0]}, Val={X_val.shape[0]}, Test={X_test.shape[0]}")
    print(f"Feature dimension: {X_train.shape[1]}")

    # Save split data for reproducibility
    split_data = {
        'X_train_shape': X_train.shape,
        'X_val_shape': X_val.shape,
        'X_test_shape': X_test.shape,
        'angles_train': angles_train.tolist() if isinstance(angles_train, np.ndarray) else angles_train,
        'angles_val': angles_val.tolist() if isinstance(angles_val, np.ndarray) else angles_val,
        'angles_test': angles_test.tolist() if isinstance(angles_test, np.ndarray) else angles_test,
        'class_distribution_train': np.sum(y_train, axis=0).tolist(),
        'class_distribution_val': np.sum(y_val, axis=0).tolist(),
        'class_distribution_test': np.sum(y_test, axis=0).tolist()
    }
    with open(os.path.join(CONFIG['results_dir'], 'data_split_info.pkl'), 'wb') as f:
        pickle.dump(split_data, f)

    # Recognition models to train with specific seeds
    models_to_train = [
        ('MLP_Recognition', build_mlp_recognition_model(X_train.shape[1:]), 111),
        ('VisionTransformer', build_vision_transformer_model(X_train.shape[1:]), 222),
        ('GaitNet', build_gaitnet_model(X_train.shape[1:]), 333)
    ]

    # Train each model
    results = {}
    for model_name, model, seed in models_to_train:
        print(f"\n{'='*50}")
        print(f"Training {model_name} model with seed {seed}...")
        print(f"{'='*50}")

        tf.random.set_seed(seed)
        np.random.seed(seed)

        try:
            history, trained_model, training_time = train_model(
                model, X_train, y_train, X_val, y_val,
                epochs=CONFIG['epochs'], batch_size=CONFIG['batch_size'],
                model_name=model_name
            )

            print(f"\nEvaluating {model_name} model...")
            accuracy, cm, metrics = evaluate_model(
                trained_model, X_test, y_test, angles_test=angles_test, model_name=model_name
            )

            results[model_name] = {
                'model': trained_model,
                'history': history,
                'accuracy': accuracy,
                'confusion_matrix': cm,
                'metrics': metrics,
                'training_time': training_time
            }

            # Save important data separately
            with open(os.path.join(CONFIG['results_dir'], f"{model_name}_results.pkl"), 'wb') as f:
                pickle.dump({
                    'accuracy': accuracy,
                    'confusion_matrix': cm,
                    'history': history.history,
                    'metrics': metrics,
                    'training_time': training_time
                }, f)
        except Exception as e:
            print(f"Error training/evaluating {model_name}: {e}")
            # Continue with next model if one fails

    # Create and save performance table with all metrics
    performance_metrics = []
    for name, info in results.items():
        metrics = info['metrics']
        performance_metrics.append({
            'Model': name,
            'Accuracy': metrics['accuracy'],
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'F1': metrics['f1'],
            'Training_Time': info['training_time']
        })

    performance_df = pd.DataFrame(performance_metrics).sort_values('Accuracy', ascending=False)
    performance_df.to_csv(os.path.join(CONFIG['results_dir'], "recognition_model_performance.csv"), index=False)

    print("\nRecognition Model Performance Ranking:")
    print(performance_df.to_string(index=False))

    # Save combined results
    with open(os.path.join(CONFIG['results_dir'], "all_recognition_results.pkl"), 'wb') as f:
        pickle.dump(results, f)

    return results, performance_df

In [8]:
# Main execution function
def run_notebook4():
    """Run Notebook 4 pipeline: Advanced Recognition Models with ResNet101 and EfficientNetB2"""
    print("="*80)
    print("NOTEBOOK 4: ADVANCED RECOGNITION MODELS WITH RESNET101 AND EFFICIENTNETB2")
    print("="*80)

    # Create a file to track the start time
    start_time = time.time()
    with open(os.path.join(CONFIG['results_dir'], 'execution_times.json'), 'w') as f:
        json.dump({'notebook4_start_time': start_time}, f)

    # Load and process dataset - direct approach
    print("\nLoading and processing dataset...")
    dataset_df, dataset_path = load_dataset_from_drive()
    dataset_load_time = time.time()

    # Build feature extractors
    print("\nBuilding feature extractors...")
    feature_extractors = build_feature_extractors()
    feature_extractor_build_time = time.time()

    # Extract features
    print("\nExtracting features...")
    features, labels, paths, angles = process_and_extract_features(
        feature_extractors, dataset_df, max_samples=CONFIG['max_samples']
    )
    feature_extraction_time = time.time()

    # Concatenate features with PCA
    print("\nConcatenating features with PCA...")
    concatenated_features, pca_model = concatenate_features(features, apply_pca=True)
    pca_time = time.time()

    # Save feature info for Notebook 5
    feature_info = {
        'feature_shape': concatenated_features.shape,
        'num_samples': concatenated_features.shape[0],
        'feature_dimension': concatenated_features.shape[1],
        'is_pca_reduced': True,
        'resnet_dim': features['resnet'].shape[1],
        'efficientnet_dim': features['efficientnet'].shape[1]
    }
    with open(os.path.join(CONFIG['results_dir'], 'feature_info.json'), 'w') as f:
        json.dump(feature_info, f)

    # Train recognition models
    print("\nTraining advanced recognition models...")
    try:
        model_results, performance_df = train_all_recognition_models(
            concatenated_features, labels, angles
        )
        model_training_time = time.time()

        # Save best model info
        best_model_name = performance_df.iloc[0]['Model'] if not performance_df.empty else "None"
        best_model_accuracy = performance_df.iloc[0]['Accuracy'] if not performance_df.empty else 0
    except Exception as e:
        print(f"Error during recognition model training: {e}")
        model_training_time = time.time()
        model_results = {}
        performance_df = pd.DataFrame()
        best_model_name = "Error"
        best_model_accuracy = 0

    # Save execution times
    execution_times = {
        'total_time': model_training_time - start_time,
        'dataset_loading_time': dataset_load_time - start_time,
        'feature_extractor_build_time': feature_extractor_build_time - dataset_load_time,
        'feature_extraction_time': feature_extraction_time - feature_extractor_build_time,
        'pca_time': pca_time - feature_extraction_time,
        'model_training_time': model_training_time - pca_time,
        'models': {name: info['training_time'] for name, info in model_results.items()} if model_results else {}
    }

    with open(os.path.join(CONFIG['results_dir'], 'execution_times.json'), 'w') as f:
        json.dump(execution_times, f)

    # Create README with results summary
    with open(os.path.join(CONFIG['results_dir'], "README.md"), 'w') as f:
        f.write("# Notebook 4: Advanced Recognition Models with ResNet101 and EfficientNetB2\n\n")
        f.write("## Dataset and Feature Extraction\n")
        f.write(f"- Sample ratio used: {CONFIG['sample_ratio']}\n")
        f.write(f"- ResNet101 feature dimension: {features['resnet'].shape[1]}\n")
        f.write(f"- EfficientNetB2 feature dimension: {features['efficientnet'].shape[1]}\n")
        f.write(f"- Final feature dimension after PCA: {concatenated_features.shape[1]}\n")
        f.write(f"- Number of samples: {concatenated_features.shape[0]}\n\n")

        if not performance_df.empty:
            f.write("## Recognition Models Performance\n")
            f.write(performance_df.to_markdown(index=False) + "\n\n")
            f.write(f"### Best Model: {best_model_name}\n")
            f.write(f"- Accuracy: {best_model_accuracy:.4f}\n")
        else:
            f.write("## Recognition Models\n")
            f.write("Error occurred during model training. See logs for details.\n\n")

        f.write(f"- Total execution time: {execution_times['total_time']:.2f} seconds\n")

    # Create a summary of data collected for Notebook 5
    notebook5_data_summary = {
        'feature_info': 'Information about input features',
        'distribution_data': 'Sample distribution by walking style and angle',
        'pca_data': 'PCA variance ratios, components, thresholds',
        'execution_times': 'Timing of each pipeline component',
        'model_metrics': 'Performance metrics for each model',
        'model_history': 'Training history for each model',
        'confusion_matrices': 'Overall and per-angle confusion matrices',
        'angle_metrics': 'Detailed performance by viewing angle'
    }

    # Save summary for easier reference
    with open(os.path.join(CONFIG['results_dir'], 'data_summary_for_notebook5.json'), 'w') as f:
        json.dump(notebook5_data_summary, f)

    print("\nNOTEBOOK 4 COMPLETED")
    return {
        'features': features,
        'concatenated_features': concatenated_features,
        'pca_model': pca_model,
        'labels': labels,
        'angles': angles,
        'paths': paths,
        'models': model_results,
        'performance': performance_df,
        'execution_times': execution_times
    }

# Run the notebook
if __name__ == "__main__":
    run_notebook4()

NOTEBOOK 4: ADVANCED RECOGNITION MODELS WITH RESNET101 AND EFFICIENTNETB2

Loading and processing dataset...
Mounted at /content/drive



Building feature extractors...
171446536/171446536 ━━━━━━━━━━━━━━━━━━━━ 8s 0us/step
31790344/31790344 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step

Extracting features...



Concatenating features with PCA...
Using 256 PCA components (max possible: 3455)

Training advanced recognition models...
Data split: Train=396469, Val=56639, Test=113278
Feature dimension: 256
Built MLP with input shape: (256,)
Built VisionTransformer with input shape: (256,)
Built GaitNet with input shape: (256,)

Training MLP_Recognition model with seed 111...
Epoch 1/32
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5432 - loss: 1.0450
Epoch 1: val_accuracy improved from -inf to 0.76852, saving model to notebook5/Notebook4Results/MLP_Recognition_best.h5


3098/3098 ━━━━━━━━━━━━━━━━━━━━ 17s 4ms/step - accuracy: 0.5432 - loss: 1.0450 - val_accuracy: 0.7685 - val_loss: 0.5786 - learning_rate: 1.0000e-04
Epoch 2/32
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7175 - loss: 0.6655
Epoch 2: val_accuracy improved from 0.76852 to 0.80081, saving model to notebook5/Notebook4Results/MLP_Recognition_best.h5


3098/3098 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.7175 - loss: 0.6655 - val_accuracy: 0.8008 - val_loss: 0.5088 - learning_rate: 1.0000e-04
Epoch 3/32
3094/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7605 - loss: 0.5719
Epoch 3: val_accuracy improved from 0.80081 to 0.81530, saving model to notebook5/Notebook4Results/MLP_Recognition_best.h5


...

3098/3098 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9113 - loss: 0.2005 - val_accuracy: 0.9085 - val_loss: 0.2353 - learning_rate: 2.0000e-04
Epoch 29/32
3089/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9128 - loss: 0.1978
Epoch 29: val_accuracy did not improve from 0.90854
3098/3098 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9128 - loss: 0.1978 - val_accuracy: 0.9082 - val_loss: 0.2353 - learning_rate: 2.0000e-04
Epoch 30/32
3088/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9138 - loss: 0.1947
Epoch 30: val_accuracy improved from 0.90854 to 0.90927, saving model to notebook5/Notebook4Results/GaitNet_best.h5


3098/3098 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9138 - loss: 0.1947 - val_accuracy: 0.9093 - val_loss: 0.2348 - learning_rate: 2.0000e-04
Epoch 31/32
3091/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9148 - loss: 0.1933
Epoch 31: val_accuracy improved from 0.90927 to 0.90967, saving model to notebook5/Notebook4Results/GaitNet_best.h5


3098/3098 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9148 - loss: 0.1933 - val_accuracy: 0.9097 - val_loss: 0.2330 - learning_rate: 2.0000e-04
Epoch 32/32
3095/3098 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9155 - loss: 0.1904
Epoch 32: val_accuracy improved from 0.90967 to 0.91077, saving model to notebook5/Notebook4Results/GaitNet_best.h5


3098/3098 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9155 - loss: 0.1904 - val_accuracy: 0.9108 - val_loss: 0.2320 - learning_rate: 2.0000e-04
Restoring model weights from the end of the best epoch: 32.



Evaluating GaitNet model...
3540/3540 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step
Model: GaitNet, Accuracy: 0.9133, Precision: 0.9160, Recall: 0.9133, F1: 0.9140
Confusion Matrix:
[[62898  3457  2380]
 [ 1868 19717   609]
 [  993   512 20844]]

Recognition Model Performance Ranking:
            Model  Accuracy  Precision   Recall       F1  Training_Time
          GaitNet  0.913319   0.916026 0.913319 0.914033     274.535456
  MLP_Recognition  0.909841   0.912306 0.909841 0.910516     275.894907
VisionTransformer  0.905436   0.909036 0.905436 0.906222     276.380781

NOTEBOOK 4 COMPLETED
